# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading

Load the dataset metadata and structure from the Croissant schema using `mlcroissant`:

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as an object (not as a dict)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview

Explore the available record sets, fields, and their `@id`s to understand the dataset structure.

In [ ]:
# List all record sets and their fields by @id and name
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record sets:\n")
for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    if hasattr(rs, 'name'):
        print(f"  Name: {getattr(rs, 'name', '')}")
    # Each record set may have 'fields' attribute
    fields = getattr(rs, 'fields', [])
    if fields:
        print("  Fields:")
        for fld in fields:
            print(f"    - {fld['@id']} : {getattr(fld, 'name', '')}")
    print("")

You can also print the first few records of a specific record set using their `@id`. For example:

In [ ]:
# Print first 3 records from each record set by @id
for rs in record_sets:
    print(f"\nRecords from record set: {rs['@id']}")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs['@id'])):
            print(rec)
            if i >= 2:
                break
    except Exception as e:
        print(f"Failed to load records: {e}")

## 3. Data Extraction

Load records into DataFrames for analysis. All references use the record set and field `@id`s obtained above.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = dict()

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record set {record_set_id} with shape {df.shape}")
        else:
            print(f"No records for record set {record_set_id}")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

if dataframes:
    # Pick the first loaded record set for preview
    first_id = next(iter(dataframes))
    print(f"\nColumns in DataFrame for record set {first_id}:")
    print(dataframes[first_id].columns.tolist())
    dataframes[first_id].head()
else:
    print("No record sets contain data.")

## 4. Exploratory Data Analysis (EDA)

Let's process numerical and categorical data fields. We'll demonstrate filtering, normalization, and grouping using the loaded DataFrame. Update variable names according to the field `@id`s discovered above.

In [ ]:
# Example: Select first non-empty DataFrame and use its numeric field for EDA

if dataframes:
    # Use the first available DataFrame
    record_set_id = first_id
    df = dataframes[record_set_id]
    
    # Try to detect a numeric field by sampling the dtypes
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        # Attempt to coerce non-null string columns to numeric
        for col in df.columns:
            try:
                converted = pd.to_numeric(df[col], errors='coerce')
                if converted.notnull().sum() > 0:
                    df[col] = converted
                    numeric_fields.append(col)
            except:
                continue
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field '@id': {numeric_field}")
        threshold = np.nanmean(df[numeric_field])
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean value):")
        print(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    else:
        print("No numeric field detected for EDA.")

    # Try grouping by a categorical field (look for non-numeric fields)
    group_fields = [col for col in df.columns if col not in numeric_fields]
    if group_fields:
        group_field = group_fields[0]
        print(f"\nGrouping by field '@id': {group_field}")
        if not filtered_df.empty and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data (mean {numeric_field}) by {group_field}:")
            print(grouped_df.head())
    else:
        print("No categorical field for grouping.")
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize the numeric field distribution and, if possible, how it varies by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if dataframes and numeric_fields:
    # Histogram of numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field}' (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group (if available)
    if group_fields and group_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"'{numeric_field}' by '{group_field}' group (@ids)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion

In this notebook, we loaded and explored a FAIR² dataset package using the `mlcroissant` library. Using entity `@id` references, we:
- Queried available record sets and their data fields.
- Loaded tables into pandas DataFrames and explored summary statistics.
- Demonstrated numeric filtering, normalization, grouping, and basic visualization.

This approach enables reproducible and semantically robust interaction with complex datasets described by Croissant schemas.